In [13]:
import os
import numpy as np
import pandas as pd 
import seaborn as sns
import requests
from bs4 import BeautifulSoup as soup
from newspaper import Article
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score , confusion_matrix
import seaborn as sns
from textblob import TextBlob
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from newspaper import Article, ArticleException
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import FunctionTransformer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import precision_recall_fscore_support

import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning)

In [153]:
import sys
import os

module_path = os.path.abspath(os.path.join('..', 'utils')) 

if module_path not in sys.path:
    sys.path.append(module_path)

from custom_utils import VaderSentimentExtractor

In [14]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\vedan\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [15]:
root = r"C:\Users\vedan\Downloads/data/"
art_list = os.listdir(root)
print(f"total articles {len(art_list)}")
print("sample titles: ")
print(*art_list[:3], sep='\n')

total articles 7204
sample titles: 
20181010_business_india-business_bob-partners-truecaller-for-upi-payments.txt
20181010_business_india-business_city-to-get-a-centre-of-excellence-for-fintech.txt
20181010_business_india-business_cos-should-offer-flexible-work-policies-to-attract-best-talent.txt


In [16]:
contents = []
for filename in art_list:

    file_path = os.path.join(root, filename)
    
  
    with open(file_path, 'r', encoding='utf-8') as f:
        
        contents.append(f.read())

In [17]:
df = pd.DataFrame(list(zip(art_list, contents)), columns=['title', 'content'])
df.head()

,title,content
0,20181010_business_india-business_bob-partners-...,[Mumbai: Bank of Baroda (BoB) has tied-up with...
1,20181010_business_india-business_city-to-get-a...,[Chennai will soon be home to a Centre of Exce...
2,20181010_business_india-business_cos-should-of...,"[By Kamal Karanth According to Aristotle, ‘The..."
3,20181010_business_india-business_deal-flows-cu...,[CHENNAI: The traditionally strong quarter (Ju...
4,20181010_business_india-business_equity-mutual...,[Coimbatore: FIIs (foreign institutional inves...


In [18]:
df['date'] = df.title.apply(lambda x: int(x.split('_')[0]))
df['tag'] = df.title.apply(lambda x: ("_".join(x.split('-')[0].split('_')[1:-1])))
df.head()

,title,content,date,tag
0,20181010_business_india-business_bob-partners-...,[Mumbai: Bank of Baroda (BoB) has tied-up with...,20181010,business
1,20181010_business_india-business_city-to-get-a...,[Chennai will soon be home to a Centre of Exce...,20181010,business
2,20181010_business_india-business_cos-should-of...,"[By Kamal Karanth According to Aristotle, ‘The...",20181010,business
3,20181010_business_india-business_deal-flows-cu...,[CHENNAI: The traditionally strong quarter (Ju...,20181010,business
4,20181010_business_india-business_equity-mutual...,[Coimbatore: FIIs (foreign institutional inves...,20181010,business


In [19]:
months = [i for i in range(1, 13)]
days = [i for i in range(1, 32)]

def convert(x):
    x = str(x)
    splits = [(int(x[:k]), int(x[k:])) for k in range(1, len(x))]
    for i, j in splits:
        if i in months and j in days: 
            return i, j

df['year'] = df.title.apply(lambda x: int(x.split('_')[0][:4]))
rest = df.title.apply(lambda x: int(x.split('_')[0][4:]))
a = rest.apply(convert)
df['month'] = [i[0] for i in a]
df['day'] = [i[1] for i in a]
df.head()

,title,content,date,tag,year,month,day
0,20181010_business_india-business_bob-partners-...,[Mumbai: Bank of Baroda (BoB) has tied-up with...,20181010,business,2018,1,10
1,20181010_business_india-business_city-to-get-a...,[Chennai will soon be home to a Centre of Exce...,20181010,business,2018,1,10
2,20181010_business_india-business_cos-should-of...,"[By Kamal Karanth According to Aristotle, ‘The...",20181010,business,2018,1,10
3,20181010_business_india-business_deal-flows-cu...,[CHENNAI: The traditionally strong quarter (Ju...,20181010,business,2018,1,10
4,20181010_business_india-business_equity-mutual...,[Coimbatore: FIIs (foreign institutional inves...,20181010,business,2018,1,10


In [20]:
df['headline'] = df.title.apply(lambda x: (x.split('-')[0].split('_')[-1] + '-' + '-'.join(x.split('-')[1:])).replace("-", " ")[:-3])
df['content'] = df.content.apply(lambda x: x[1:-1])
df.head()

,title,content,date,tag,year,month,day,headline
0,20181010_business_india-business_bob-partners-...,Mumbai: Bank of Baroda (BoB) has tied-up with ...,20181010,business,2018,1,10,india business_bob partners truecaller for upi...
1,20181010_business_india-business_city-to-get-a...,Chennai will soon be home to a Centre of Excel...,20181010,business,2018,1,10,india business_city to get a centre of excelle...
2,20181010_business_india-business_cos-should-of...,"By Kamal Karanth According to Aristotle, ‘The ...",20181010,business,2018,1,10,india business_cos should offer flexible work ...
3,20181010_business_india-business_deal-flows-cu...,CHENNAI: The traditionally strong quarter (Jul...,20181010,business,2018,1,10,india business_deal flows currency move to hel...
4,20181010_business_india-business_equity-mutual...,Coimbatore: FIIs (foreign institutional invest...,20181010,business,2018,1,10,india business_equity mutual funds remain bull...


In [21]:
def get_loc(x):
    p = x.split(':')[0]
    if len(p.split(" ")) < 6:
        return p
    elif len(p.split('()')[0]) < 30:
        return p.split(',')[0]
    return ""

df['loc'] = df['content'].apply(get_loc)
df.head()

,title,content,date,tag,year,month,day,headline,loc
0,20181010_business_india-business_bob-partners-...,Mumbai: Bank of Baroda (BoB) has tied-up with ...,20181010,business,2018,1,10,india business_bob partners truecaller for upi...,Mumbai
1,20181010_business_india-business_city-to-get-a...,Chennai will soon be home to a Centre of Excel...,20181010,business,2018,1,10,india business_city to get a centre of excelle...,
2,20181010_business_india-business_cos-should-of...,"By Kamal Karanth According to Aristotle, ‘The ...",20181010,business,2018,1,10,india business_cos should offer flexible work ...,
3,20181010_business_india-business_deal-flows-cu...,CHENNAI: The traditionally strong quarter (Jul...,20181010,business,2018,1,10,india business_deal flows currency move to hel...,CHENNAI
4,20181010_business_india-business_equity-mutual...,Coimbatore: FIIs (foreign institutional invest...,20181010,business,2018,1,10,india business_equity mutual funds remain bull...,Coimbatore


In [22]:
df = df[['date','year','month' ,'day', 'tag', 'loc', 'headline', 'title', 'content']]
df.head()

,date,year,month,day,tag,loc,headline,title,content
0,20181010,2018,1,10,business,Mumbai,india business_bob partners truecaller for upi...,20181010_business_india-business_bob-partners-...,Mumbai: Bank of Baroda (BoB) has tied-up with ...
1,20181010,2018,1,10,business,,india business_city to get a centre of excelle...,20181010_business_india-business_city-to-get-a...,Chennai will soon be home to a Centre of Excel...
2,20181010,2018,1,10,business,,india business_cos should offer flexible work ...,20181010_business_india-business_cos-should-of...,"By Kamal Karanth According to Aristotle, ‘The ..."
3,20181010,2018,1,10,business,CHENNAI,india business_deal flows currency move to hel...,20181010_business_india-business_deal-flows-cu...,CHENNAI: The traditionally strong quarter (Jul...
4,20181010,2018,1,10,business,Coimbatore,india business_equity mutual funds remain bull...,20181010_business_india-business_equity-mutual...,Coimbatore: FIIs (foreign institutional invest...


In [23]:
def get_sentiment(text):
    if text == '':
        return 0.0, 0.0
    analysis = TextBlob(text)
    return analysis.sentiment.polarity, analysis.sentiment.subjectivity

In [24]:
df[['polarity', 'subjectivity']] = df['content'].apply(
    lambda x: pd.Series(get_sentiment(x))
)

In [25]:
df

,date,year,month,day,tag,loc,headline,title,content,polarity,subjectivity
0,20181010,2018,1,10,business,Mumbai,india business_bob partners truecaller for upi...,20181010_business_india-business_bob-partners-...,Mumbai: Bank of Baroda (BoB) has tied-up with ...,0.293290,0.451491
1,20181010,2018,1,10,business,,india business_city to get a centre of excelle...,20181010_business_india-business_city-to-get-a...,Chennai will soon be home to a Centre of Excel...,-0.026154,0.355000
2,20181010,2018,1,10,business,,india business_cos should offer flexible work ...,20181010_business_india-business_cos-should-of...,"By Kamal Karanth According to Aristotle, ‘The ...",0.213180,0.475056
3,20181010,2018,1,10,business,CHENNAI,india business_deal flows currency move to hel...,20181010_business_india-business_deal-flows-cu...,CHENNAI: The traditionally strong quarter (Jul...,0.102381,0.409524
4,20181010,2018,1,10,business,Coimbatore,india business_equity mutual funds remain bull...,20181010_business_india-business_equity-mutual...,Coimbatore: FIIs (foreign institutional invest...,0.133001,0.397082
...,...,...,...,...,...,...,...,...,...,...,...
7199,20181027,2018,1,27,city_pune,PUNE,alert residents nab duo for robbing carpenter.,20181027_city_pune_alert-residents-nab-duo-for...,PUNE: Alert residents nabbed a man and his min...,0.021726,0.346726
7200,20181027,2018,1,27,city_pune,PUNE,as festival nears prices of bajra wheat jowar ...,20181027_city_pune_as-festival-nears-prices-of...,"PUNE: After onions, it’s now the turn of wheat...",0.024463,0.341648
7201,20181027,2018,1,27,city_pune,PUNE,eco friendly twist to this years lanterns for ...,20181027_city_pune_eco-friendly-twist-to-this-...,PUNE: Shops lined in the Raviwar Peth market a...,0.153783,0.447407
7202,20181027,2018,1,27,city_pune,PUNE,ferreira gonsalves back in police custody afte...,20181027_city_pune_ferreira-gonsalves-back-in-...,PUNE: The city police on Friday evening took a...,-0.005102,0.302551


In [28]:
SUBJECTIVITY_THRESHOLD = 0.5
df['bias_label'] = (df['subjectivity'] > SUBJECTIVITY_THRESHOLD).astype(int)

In [29]:
df

,date,year,month,day,tag,loc,headline,title,content,polarity,subjectivity,bias_label
0,20181010,2018,1,10,business,Mumbai,india business_bob partners truecaller for upi...,20181010_business_india-business_bob-partners-...,Mumbai: Bank of Baroda (BoB) has tied-up with ...,0.293290,0.451491,0
1,20181010,2018,1,10,business,,india business_city to get a centre of excelle...,20181010_business_india-business_city-to-get-a...,Chennai will soon be home to a Centre of Excel...,-0.026154,0.355000,0
2,20181010,2018,1,10,business,,india business_cos should offer flexible work ...,20181010_business_india-business_cos-should-of...,"By Kamal Karanth According to Aristotle, ‘The ...",0.213180,0.475056,0
3,20181010,2018,1,10,business,CHENNAI,india business_deal flows currency move to hel...,20181010_business_india-business_deal-flows-cu...,CHENNAI: The traditionally strong quarter (Jul...,0.102381,0.409524,0
4,20181010,2018,1,10,business,Coimbatore,india business_equity mutual funds remain bull...,20181010_business_india-business_equity-mutual...,Coimbatore: FIIs (foreign institutional invest...,0.133001,0.397082,0
...,...,...,...,...,...,...,...,...,...,...,...,...
7199,20181027,2018,1,27,city_pune,PUNE,alert residents nab duo for robbing carpenter.,20181027_city_pune_alert-residents-nab-duo-for...,PUNE: Alert residents nabbed a man and his min...,0.021726,0.346726,0
7200,20181027,2018,1,27,city_pune,PUNE,as festival nears prices of bajra wheat jowar ...,20181027_city_pune_as-festival-nears-prices-of...,"PUNE: After onions, it’s now the turn of wheat...",0.024463,0.341648,0
7201,20181027,2018,1,27,city_pune,PUNE,eco friendly twist to this years lanterns for ...,20181027_city_pune_eco-friendly-twist-to-this-...,PUNE: Shops lined in the Raviwar Peth market a...,0.153783,0.447407,0
7202,20181027,2018,1,27,city_pune,PUNE,ferreira gonsalves back in police custody afte...,20181027_city_pune_ferreira-gonsalves-back-in-...,PUNE: The city police on Friday evening took a...,-0.005102,0.302551,0


In [30]:
analyzer = SentimentIntensityAnalyzer()

VADER_THRESHOLD = 0.5 
def relabel_with_vader(content):
    if not isinstance(content, str):
        return 0
    scores = analyzer.polarity_scores(content)
    compound_score = scores['compound']
    
    if abs(compound_score) >= VADER_THRESHOLD:
        return 1
    
    return 0 #0 for unbiased and 1 for biased
df['bias_label_vader'] = df['content'].apply(relabel_with_vader)

print(f"New Target Class Counts (df['bias_label_vader']):\n{df['bias_label_vader'].value_counts()}")

New Target Class Counts (df['bias_label_vader']):
bias_label_vader
1    6245
0     959
Name: count, dtype: int64


In [31]:
df

,date,year,month,day,tag,loc,headline,title,content,polarity,subjectivity,bias_label,bias_label_vader
0,20181010,2018,1,10,business,Mumbai,india business_bob partners truecaller for upi...,20181010_business_india-business_bob-partners-...,Mumbai: Bank of Baroda (BoB) has tied-up with ...,0.293290,0.451491,0,1
1,20181010,2018,1,10,business,,india business_city to get a centre of excelle...,20181010_business_india-business_city-to-get-a...,Chennai will soon be home to a Centre of Excel...,-0.026154,0.355000,0,1
2,20181010,2018,1,10,business,,india business_cos should offer flexible work ...,20181010_business_india-business_cos-should-of...,"By Kamal Karanth According to Aristotle, ‘The ...",0.213180,0.475056,0,1
3,20181010,2018,1,10,business,CHENNAI,india business_deal flows currency move to hel...,20181010_business_india-business_deal-flows-cu...,CHENNAI: The traditionally strong quarter (Jul...,0.102381,0.409524,0,1
4,20181010,2018,1,10,business,Coimbatore,india business_equity mutual funds remain bull...,20181010_business_india-business_equity-mutual...,Coimbatore: FIIs (foreign institutional invest...,0.133001,0.397082,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7199,20181027,2018,1,27,city_pune,PUNE,alert residents nab duo for robbing carpenter.,20181027_city_pune_alert-residents-nab-duo-for...,PUNE: Alert residents nabbed a man and his min...,0.021726,0.346726,0,1
7200,20181027,2018,1,27,city_pune,PUNE,as festival nears prices of bajra wheat jowar ...,20181027_city_pune_as-festival-nears-prices-of...,"PUNE: After onions, it’s now the turn of wheat...",0.024463,0.341648,0,1
7201,20181027,2018,1,27,city_pune,PUNE,eco friendly twist to this years lanterns for ...,20181027_city_pune_eco-friendly-twist-to-this-...,PUNE: Shops lined in the Raviwar Peth market a...,0.153783,0.447407,0,0
7202,20181027,2018,1,27,city_pune,PUNE,ferreira gonsalves back in police custody afte...,20181027_city_pune_ferreira-gonsalves-back-in-...,PUNE: The city police on Friday evening took a...,-0.005102,0.302551,0,1


In [32]:
X = df[['content']].copy()
y = df['bias_label_vader']

In [33]:
valid_mask = X['content'].notna() & (X['content'].apply(lambda x: isinstance(x, str))) & (X['content'].str.len() > 0)

In [34]:
X_cleaned = X[valid_mask].copy()
y_cleaned = y[valid_mask].copy()

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X_cleaned, y_cleaned, test_size=0.2, random_state=42, stratify=y_cleaned
)

In [37]:
print("X_train type:", type(X_train))
print("X_train columns:", X_train.columns.tolist())

X_train type: <class 'pandas.core.frame.DataFrame'>
X_train columns: ['content']


In [157]:
from custom_utils import VaderSentimentExtractor 
from custom_utils import sparse_to_dense

In [158]:
df

,date,year,month,day,tag,loc,headline,title,content,polarity,subjectivity,bias_label,bias_label_vader
0,20181010,2018,1,10,business,Mumbai,india business_bob partners truecaller for upi...,20181010_business_india-business_bob-partners-...,Mumbai: Bank of Baroda (BoB) has tied-up with ...,0.293290,0.451491,0,1
1,20181010,2018,1,10,business,,india business_city to get a centre of excelle...,20181010_business_india-business_city-to-get-a...,Chennai will soon be home to a Centre of Excel...,-0.026154,0.355000,0,1
2,20181010,2018,1,10,business,,india business_cos should offer flexible work ...,20181010_business_india-business_cos-should-of...,"By Kamal Karanth According to Aristotle, ‘The ...",0.213180,0.475056,0,1
3,20181010,2018,1,10,business,CHENNAI,india business_deal flows currency move to hel...,20181010_business_india-business_deal-flows-cu...,CHENNAI: The traditionally strong quarter (Jul...,0.102381,0.409524,0,1
4,20181010,2018,1,10,business,Coimbatore,india business_equity mutual funds remain bull...,20181010_business_india-business_equity-mutual...,Coimbatore: FIIs (foreign institutional invest...,0.133001,0.397082,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7199,20181027,2018,1,27,city_pune,PUNE,alert residents nab duo for robbing carpenter.,20181027_city_pune_alert-residents-nab-duo-for...,PUNE: Alert residents nabbed a man and his min...,0.021726,0.346726,0,1
7200,20181027,2018,1,27,city_pune,PUNE,as festival nears prices of bajra wheat jowar ...,20181027_city_pune_as-festival-nears-prices-of...,"PUNE: After onions, it’s now the turn of wheat...",0.024463,0.341648,0,1
7201,20181027,2018,1,27,city_pune,PUNE,eco friendly twist to this years lanterns for ...,20181027_city_pune_eco-friendly-twist-to-this-...,PUNE: Shops lined in the Raviwar Peth market a...,0.153783,0.447407,0,0
7202,20181027,2018,1,27,city_pune,PUNE,ferreira gonsalves back in police custody afte...,20181027_city_pune_ferreira-gonsalves-back-in-...,PUNE: The city police on Friday evening took a...,-0.005102,0.302551,0,1


In [159]:
to_dense_transformer = FunctionTransformer(
    sparse_to_dense,
    accept_sparse=True 
)

In [160]:
preprocessor = ColumnTransformer(
    transformers=[
        ('text_pipeline', TfidfVectorizer(max_features=4000, stop_words='english', ngram_range=(1, 2)), 'content'),
        ('vader_pipeline', Pipeline([
            ('vader_extractor', VaderSentimentExtractor()),
            ('scaler', StandardScaler())
        ]), 'content')
    ],
    remainder='drop'
)


In [161]:
preprocessor

ColumnTransformer(transformers=[('text_pipeline',
                                 TfidfVectorizer(max_features=4000,
                                                 ngram_range=(1, 2),
                                                 stop_words='english'),
                                 'content'),
                                ('vader_pipeline',
                                 Pipeline(steps=[('vader_extractor',
                                                  VaderSentimentExtractor()),
                                                 ('scaler', StandardScaler())]),
                                 'content')])

In [162]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), 
    ('to_dense', to_dense_transformer),
    ('classifier', HistGradientBoostingClassifier(
        max_iter=500, 
        random_state=42
    )) 
])

In [163]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text_pipeline',
                                                  TfidfVectorizer(max_features=4000,
                                                                  ngram_range=(1,
                                                                               2),
                                                                  stop_words='english'),
                                                  'content'),
                                                 ('vader_pipeline',
                                                  Pipeline(steps=[('vader_extractor',
                                                                   VaderSentimentExtractor()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  'content')])),
                ('to_dense',
                 FunctionTransformer(accept_sparse=True,
                                     func=<function sparse_to_dense at 0x00000209503C5EE0>)),
                ('classifier',
                 HistGradientBoostingClassifier(max_iter=500,
                                                random_state=42))])

In [164]:
y_pred_final = pipeline.predict(X_test)

In [165]:
accuracy_score(y_test,y_pred_final)

0.9979181124219292

In [166]:
print(classification_report(y_test, y_pred_final, target_names=['Unbiased (0)', 'Biased (1)']))

              precision    recall  f1-score   support

Unbiased (0)       1.00      0.98      0.99       191
  Biased (1)       1.00      1.00      1.00      1250

    accuracy                           1.00      1441
   macro avg       1.00      0.99      1.00      1441
weighted avg       1.00      1.00      1.00      1441



In [167]:
cm_fixed = confusion_matrix(y_test, y_pred_final)
print("\nConfusion Matrix:")
print(pd.DataFrame(cm_fixed, index=['True Unbiased', 'True Biased'], columns=['Predicted Unbiased', 'Predicted Biased']).to_markdown(numalign="left", stralign="left"))


Confusion Matrix:
|               | Predicted Unbiased   | Predicted Biased   |
|:--------------|:---------------------|:-------------------|
| True Unbiased | 188                  | 3                  |
| True Biased   | 0                    | 1250               |


In [168]:
def predict_bias_pipeline(raw_articles, trained_pipeline):
    
    X_new = pd.DataFrame({'content': raw_articles})
    
    predictions = trained_pipeline.predict(X_new)
    probabilities = trained_pipeline.predict_proba(X_new)[:, 1] 
    
    print("\nPredicted Probabilities of being Biased (1):")
    for i, prob in enumerate(probabilities):
        print(f"Article {i+1}: {prob:.4f}")
    
    return predictions

In [169]:
new_articles = [
    """The latest quarterly earnings report shows a revenue increase of 3.2% year-over-year, 
    meeting analyst expectations. The CEO attributed the growth to increased efficiency.""",
    
    """It is utterly tragic that the company's pathetic revenue growth of a mere 3.2% 
    is being spun as a success. This clearly demonstrates management's disastrous failure 
    to innovate and their total lack of ambition. The future looks grim!""",
    
    """Yesterday, the City Council voted 4-3 to approve the new zoning ordinance in sector B-7."""
]

In [170]:
 predictions = predict_bias_pipeline(new_articles, pipeline)


Predicted Probabilities of being Biased (1):
Article 1: 1.0000
Article 2: 1.0000
Article 3: 0.0000


In [171]:
print("\nFinal Predicted Labels:", predictions)


Final Predicted Labels: [1 1 0]


In [172]:
probabilities = pipeline.predict_proba(X_test)[:, 1] 

In [173]:
thresholds = np.arange(0.5, 0.90, 0.02) # Check thresholds from 0.5 to 0.88
best_f1 = 0
best_threshold = 0.5

In [174]:
print("--- Threshold Optimization Results ---")
print("| Threshold | Accuracy | Precision | Recall | F1-Score |")
print("|-----------|----------|-----------|--------|----------|")
for t in thresholds:
    y_pred_new = (probabilities >= t).astype(int)
    
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred_new, average='binary', pos_label=1)
    accuracy = accuracy_score(y_test, y_pred_new)
    
    print(f"| {t:.2f}      | {accuracy:.4f}   | {precision:.4f}  | {recall:.4f} | {f1:.4f}   |")
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print(f"\nOptimal Threshold (Max F1-Score): {best_threshold:.4f}")

--- Threshold Optimization Results ---
| Threshold | Accuracy | Precision | Recall | F1-Score |
|-----------|----------|-----------|--------|----------|
| 0.50      | 0.9979   | 0.9976  | 1.0000 | 0.9988   |
| 0.52      | 0.9979   | 0.9976  | 1.0000 | 0.9988   |
| 0.54      | 0.9979   | 0.9976  | 1.0000 | 0.9988   |
| 0.56      | 0.9979   | 0.9976  | 1.0000 | 0.9988   |
| 0.58      | 0.9979   | 0.9976  | 1.0000 | 0.9988   |
| 0.60      | 0.9979   | 0.9976  | 1.0000 | 0.9988   |
| 0.62      | 0.9979   | 0.9976  | 1.0000 | 0.9988   |
| 0.64      | 0.9979   | 0.9976  | 1.0000 | 0.9988   |
| 0.66      | 0.9979   | 0.9976  | 1.0000 | 0.9988   |
| 0.68      | 0.9979   | 0.9976  | 1.0000 | 0.9988   |
| 0.70      | 0.9979   | 0.9976  | 1.0000 | 0.9988   |
| 0.72      | 0.9979   | 0.9976  | 1.0000 | 0.9988   |
| 0.74      | 0.9979   | 0.9976  | 1.0000 | 0.9988   |
| 0.76      | 0.9979   | 0.9976  | 1.0000 | 0.9988   |
| 0.78      | 0.9979   | 0.9976  | 1.0000 | 0.9988   |
| 0.80      | 0.9979  

In [175]:
y_pred_optimized = (pipeline.predict_proba(X_test)[:, 1] >= best_threshold).astype(int)

In [176]:
def predict_bias_optimized(raw_articles, trained_pipeline, optimal_threshold):
   
    X_new = pd.DataFrame({'content': raw_articles})
    
    probabilities = trained_pipeline.predict_proba(X_new)[:, 1] 
    
    predictions = (probabilities >= optimal_threshold).astype(int)
    
    return predictions, probabilities

In [177]:
OPTIMAL_THRESHOLD_VALUE = 0.8 

In [178]:
new_articles_set = [
    # Article A: Factual Report (Expected: Unbiased / Low Probability)
    """The company announced a 3% decrease in profit for the third quarter, 
    aligning with the revised expectations provided by management last month.""",
    
    # Article B: Emotional Critique (Expected: Biased / High Probability)
    """The company's catastrophic 3% plunge in profits is a monumental failure 
    that reveals utter incompetence at the executive level and warrants immediate 
    termination of the entire leadership team!""",
    
    # Article C: Standard Business Update (Expected: Unbiased / Low Probability)
    """Oil futures closed slightly lower today amidst cautious trading 
    ahead of a major OPEC meeting scheduled for next week."""
]

In [179]:
optimized_predictions, raw_probabilities = predict_bias_optimized(
    new_articles_set, 
    pipeline,
    OPTIMAL_THRESHOLD_VALUE
)

In [180]:
optimized_predictions, raw_probabilities

(array([0, 1, 0]), array([6.20245597e-08, 9.99999830e-01, 2.10313882e-07]))

In [181]:
labels = {0: 'Unbiased (0)', 1: 'Biased (1)'}

In [182]:
Predicted_Label = [labels[p] for p in optimized_predictions]

In [183]:
Predicted_Label

['Unbiased (0)', 'Biased (1)', 'Unbiased (0)']

In [184]:
import joblib
filename = 'media_bias_prediction_model.joblib'

In [185]:
joblib.dump(pipeline, filename)

['media_bias_prediction_model.joblib']